In [1]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

dataset = "hopper-medium-expert-v2"
if "halfcheetah" in dataset:
    env_name = "HalfCheetah-v4"
elif "hopper" in dataset:
    env_name = "Hopper-v4"
elif "walker" in dataset:
    env_name = "Walker2d-v4"

#env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v5"
print(env_name)

pkl_path = f"../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{dataset}.pkl"

def convert_raw_episode(raw_ep):
    # Convert raw observations to a NumPy array and then to a list of individual observations.
    observations = np.array(raw_ep["observations"])

    # Ensure actions and rewards are NumPy arrays.
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    # For rewards, ensure they have an extra dimension (T, 1)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # Use the last element of "terminals" as the terminated flag.
    terminals = raw_ep["terminals"]
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)

/home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hopper-v4


In [2]:
type(episodes)
from sklearn.model_selection import train_test_split

dt_train_episodes,fqe_train_episodes = train_test_split(episodes,test_size=0.3,random_state=42)

print(len(dt_train_episodes))
print(len(fqe_train_episodes))

2249
964


In [ ]:
print(f"Loaded {len(episodes)} episodes.")

from d3rlpy.dataset import ReplayBuffer, FIFOBuffer

buffer_impl = FIFOBuffer(limit=10000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)

import gymnasium as gym
import d3rlpy
import argparse
# parser.add_argument("--dataset", type=str, default="hopper-medium-v0")
# parser.add_argument("--seed", type=int, default=1)
# parser.add_argument("--gpu", type=int)
# parser.add_argument("--compile", action="store_true")
# args = parser.parse_args()

args = argparse.Namespace()
args.dataset = dataset
args.seed = 1
args.gpu = 1
args.compile = False

env = gym.make(env_name)

#dataset, env = d3rlpy.datasets.get_dataset(args.dataset)

# fix seed
d3rlpy.seed(args.seed)
d3rlpy.envs.seed_env(env, args.seed)

if "halfcheetah" in args.dataset:
    target_return = 6000
elif "hopper" in args.dataset:
    target_return = 3600
elif "walker" in args.dataset:
    target_return = 5000
else:
    raise ValueError("unsupported dataset")

Loaded 3213 episodes.
2025-07-18 11:51.37 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-07-18 11:51.37 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-07-18 11:51.37 [info     ] Action size has been automatically determined. action_size=3
Compiling /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx because it changed.
[1/1] Cythonizing /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx


performance hint: /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx:67:0: Exception check on 'c_warning_callback' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'c_warning_callback' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'c_warning_callback' to allow an error code to be returned.
performance hint: /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx:104:0: Exception check on 'c_error_callback' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'c_error_callback' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'c_error_callback' to allow an error code to be returned.

Error compiling Cython file:
-------------------------------------

CompileError: /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx

In [ ]:
import d3rlpy
from d3rlpy.dataset import InfiniteBuffer, ReplayBuffer
from d3rlpy.algos.transformer.decision_transformer import DTConstantRTGforFQE,DecisionTransformer
from d3rlpy.ope.fqe import FQE,FQEConfig
import time
start_time = time.time()


model_path = "/gpfs/data/fs72297/jklotz/programming/cloned_repos/forked_repos_for_master_thesis/d3rlpy/experiments/exp01_original_dt/slurm_files/d3rlpy_logs/gpu_array/DT_hopper-medium-expert-v2_1_20250710202727/model_epoch_12.d3"

device = "cuda:0"
dt_algo = d3rlpy.load_learnable(model_path,device=device)
fqe = FQE(algo=DTConstantRTGforFQE(dt_algo, target_return=target_return),config=FQEConfig(),device=device)
fqe.fit(replay_buffer,n_steps=200,n_steps_per_epoch=100)

end_time = time.time()
elapsed = end_time - start_time

# Print or log nicely
print(f"\nTotal runtime: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

RuntimeError: CUDA error: invalid device ordinal
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [27]:
import d3rlpy
from d3rlpy.dataset import InfiniteBuffer, ReplayBuffer
from d3rlpy.algos.transformer.decision_transformer import DTConstantRTGforFQE,DecisionTransformer
from d3rlpy.ope.fqe import FQE,FQEConfig
import time
start_time = time.time()


model_path = "/gpfs/data/fs72297/jklotz/programming/cloned_repos/forked_repos_for_master_thesis/d3rlpy/experiments/exp01_original_dt/slurm_files/d3rlpy_logs/gpu_array/DT_hopper-medium-expert-v2_1_20250710202727/model_epoch_12.d3"
#model_path = "/gpfs/data/fs72297/jklotz/programming/cloned_repos/forked_repos_for_master_thesis/d3rlpy/experiments/exp01_original_dt/d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250709180450/model_epoch_12.d3"
#params_path = "/gpfs/data/fs72297/jklotz/programming/cloned_repos/forked_repos_for_master_thesis/d3rlpy/experiments/exp01_original_dt/slurm_files/d3rlpy_logs/gpu_array/DT_hopper-medium-expert-v2_1_20250710202727/params.json"


device = "cpu"
dt_algo = d3rlpy.load_learnable(model_path,device=device)
print(type(dt_algo_loaded))
fqe = FQE(algo=DTConstantRTGforFQE(dt_algo, target_return=target_return),config=FQEConfig(),device=device)
fqe.fit(replay_buffer,n_steps=200,n_steps_per_epoch=100)

end_time = time.time()
elapsed = end_time - start_time

# Print or log nicely
print(f"\nTotal runtime: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

<class 'd3rlpy.algos.transformer.decision_transformer.DecisionTransformer'>
2025-07-17 13:29.08 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-07-17 13:29.08 [debug    ] Building models...            
2025-07-17 13:29.08 [debug    ] Models have been built.       
2025-07-17 13:29.08 [info     ] Directory is created at d3rlpy_logs/FQE_20250717132908
2025-07-17 13:29.08 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'fqe', 'params': {'batch_size': 100, 'gamma': 0.99, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'none', 'params': {}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': 

Epoch 1/2: 100%|██████████| 100/100 [00:38<00:00,  2.62it/s, loss=3.95]

2025-07-17 13:29.46 [info     ] FQE_20250717132908: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.0051052021980285646, 'time_algorithm_update': 0.37522021770477293, 'loss': 3.6762088072299957, 'time_step': 0.38040399074554443} step=100
2025-07-17 13:29.46 [info     ] Model parameters are saved to d3rlpy_logs/FQE_20250717132908/model_100.d3



Epoch 2/2: 100%|██████████| 100/100 [00:38<00:00,  2.62it/s, loss=2.52]

2025-07-17 13:30.25 [info     ] FQE_20250717132908: epoch=2 step=200 epoch=2 metrics={'time_sample_batch': 0.005015814304351806, 'time_algorithm_update': 0.37647208929061887, 'loss': 2.35789054274559, 'time_step': 0.38156620025634763} step=200
2025-07-17 13:30.25 [info     ] Model parameters are saved to d3rlpy_logs/FQE_20250717132908/model_200.d3

Total runtime: 76.38 seconds (1.27 minutes)


In [8]:
import numpy as np

init_obs = np.stack([ep.observations[0] for ep in replay_buffer.episodes])
print(init_obs.shape)      # (Nₑpisodes, *obs_shape*)


(3213, 11)


In [14]:
dt_algo._config

DecisionTransformerConfig(batch_size=64, gamma=0.99, observation_scaler=StandardObservationScaler(mean=array([ 1.32971095, -0.09838471, -0.5444255 , -0.10193573,  0.02277503,
        2.35705296, -0.06349265, -0.00374045, -0.17663514, -0.11863036,
       -0.12097615]), std=array([0.17019276, 0.05163579, 0.18159172, 0.16427722, 0.60257397,
       0.77445379, 1.49944391, 0.74881494, 1.79670128, 2.05475252,
       5.73154579]), eps=0.001), action_scaler=None, reward_scaler=MultiplyRewardScaler(multiplier=0.001), compile_graph=False, context_size=20, max_timestep=1000, learning_rate=0.0001, encoder_factory=VectorEncoderFactory(hidden_units=[128], activation='relu', use_batch_norm=False, use_layer_norm=False, dropout_rate=None, exclude_last_activation=True, last_activation=None), optim_factory=AdamWFactory(clip_grad_norm=0.25, lr_scheduler_factory=WarmupSchedulerFactory(warmup_steps=10000), betas=(0.9, 0.999), eps=1e-08, weight_decay=0.0001, amsgrad=False), num_heads=1, num_layers=3, attn_dr

In [17]:
from d3rlpy.algos.transformer.inputs import TransformerInput

In [31]:
A = dt_algo.impl.action_size
actions = np.empty((init_obs.shape[0], A), dtype=np.float32)

for i in range(init_obs.shape[0]):
    obs_i = [o[i : i + 1] for o in init_obs] if isinstance(init_obs, (list, tuple)) else init_obs[i : i + 1]
    
    actions[i] = dt_algo.predict(
        TransformerInput(
            observations   = obs_i,
            actions        = np.zeros((1, A), dtype=np.float32),
            rewards        = np.zeros((1, 1), dtype=np.float32),
            returns_to_go  = np.full((1, 1), target_return, dtype=np.float32),
            timesteps      = np.zeros(1, dtype=np.int64),
        )
    )

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [28]:
init_val  = fqe.predict_value(init_obs, actions)    # (Nₑpisodes,)
print("Mean Vπ(s₀):", init_val.mean())

Mean Vπ(s₀): 10.569408
